# Chapman-Shaoxing Preprocessing — Matched

Subsamples Chapman-Native to match PTB-XL prevalence and sample size, enabling causal isolation of prevalence as a variable in cross-dataset comparison.

## Purpose

Without this, any difference between PTB-XL and Chapman-Native results could be attributed to either:
1. Class prevalence (PTB-XL ≈7.27% vs Chapman ≈21%)
2. Sample size (PTB-XL ≈21K vs Chapman ≈45K)
3. Other dataset properties (signal quality, demographics, recording protocols)

Chapman-Matched holds (1) and (2) constant against PTB-XL, so any remaining difference must come from (3). Conversely, if Chapman-Matched results match Chapman-Native results but differ from PTB-XL, prevalence and sample size are not the explanation.

## Method

1. Load Chapman-Native features
2. Compute target sample size and prevalence to match PTB-XL
3. Stratified-sample at the patient level — keep all available AF cases until target prevalence is achieved, then sample non-AF cases to fill remainder
4. Reassign folds 1–10 stratified by AF
5. Save to `data/chapman_matched_features.csv`

**Reproducibility:** uses seeded sampling (seed=2026) so the matched cohort is deterministic across reruns.

## 1. Imports and configuration

In [ ]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Paths
PTBXL_FEATURES = Path('data/ptbxl_features.csv')
CHAPMAN_NATIVE_FEATURES = Path('data/chapman_native_features.csv')
OUTPUT_PATH = Path('data/chapman_matched_features.csv')

# Match parameters
SAMPLE_SEED = 2026
FOLD_SEED = 2026
N_FOLDS = 10

# Tolerance: prevalence within ±0.5 percentage points
PREVALENCE_TOLERANCE = 0.005
# Tolerance: sample size within ±2%
SAMPLE_SIZE_TOLERANCE = 0.02

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('chapman_matched')

## 2. Load PTB-XL and Chapman-Native to determine match parameters

In [ ]:
assert PTBXL_FEATURES.exists(), f'PTB-XL features not found: {PTBXL_FEATURES}'
assert CHAPMAN_NATIVE_FEATURES.exists(), f'Chapman-Native features not found: {CHAPMAN_NATIVE_FEATURES}'

df_ptb = pd.read_csv(PTBXL_FEATURES)
df_chap = pd.read_csv(CHAPMAN_NATIVE_FEATURES)

n_ptb = len(df_ptb)
prev_ptb = df_ptb['af_detected'].mean()
log.info(f'PTB-XL: {n_ptb} records, AF prevalence = {prev_ptb:.4f}')

n_chap = len(df_chap)
prev_chap = df_chap['af_detected'].mean()
log.info(f'Chapman-Native: {n_chap} records, AF prevalence = {prev_chap:.4f}')

n_chap_af = (df_chap['af_detected'] == 1).sum()
n_chap_nonaf = (df_chap['af_detected'] == 0).sum()
log.info(f'Chapman-Native: {n_chap_af} AF, {n_chap_nonaf} non-AF')

14:43:37 [INFO] PTB-XL: 21799 records, AF prevalence = 0.0720
14:43:37 [INFO] Chapman-Native: 45152 records, AF prevalence = 0.2179
14:43:37 [INFO] Chapman-Native: 9840 AF, 35312 non-AF


## 3. Determine target match parameters

We want N_target = n_ptb and prevalence_target = prev_ptb.
- **Target AF count:** round(n_ptb × prev_ptb)
- **Target non-AF count:** n_ptb − target AF count

Need to verify Chapman-Native has enough records of each class to fulfill this. Since Chapman has ~21% prevalence and we're targeting 7.27%, the bottleneck is non-AF: we need n_ptb × (1 − prev_ptb) ≈ 19,580 non-AF Chapman records, against ~36K available. Plenty.

In [ ]:
target_n = n_ptb
target_prevalence = prev_ptb
target_n_af = int(round(target_n * target_prevalence))
target_n_nonaf = target_n - target_n_af

log.info(f'Target: {target_n} records ({target_n_af} AF + {target_n_nonaf} non-AF)')
log.info(f'Target prevalence: {target_n_af / target_n:.4f} (matches PTB-XL {prev_ptb:.4f})')

# Capacity check
assert n_chap_af >= target_n_af, (
    f'Chapman has only {n_chap_af} AF records, need {target_n_af}. '
    f'Cannot match PTB-XL prevalence at target sample size.'
)
assert n_chap_nonaf >= target_n_nonaf, (
    f'Chapman has only {n_chap_nonaf} non-AF records, need {target_n_nonaf}.'
)
log.info('Capacity check passed: enough Chapman records to construct matched cohort')

14:43:37 [INFO] Target: 21799 records (1570 AF + 20229 non-AF)
14:43:37 [INFO] Target prevalence: 0.0720 (matches PTB-XL 0.0720)
14:43:37 [INFO] Capacity check passed: enough Chapman records to construct matched cohort


## 4. Stratified subsample

Sample with replacement = False from each class. Since each Chapman record corresponds to a unique patient, this is automatically patient-disjoint.

In [ ]:
rng = np.random.default_rng(SAMPLE_SEED)

# Indices by class
af_idx = df_chap.index[df_chap['af_detected'] == 1].tolist()
nonaf_idx = df_chap.index[df_chap['af_detected'] == 0].tolist()

# Sample without replacement
sampled_af = rng.choice(af_idx, size=target_n_af, replace=False)
sampled_nonaf = rng.choice(nonaf_idx, size=target_n_nonaf, replace=False)

sampled_indices = np.concatenate([sampled_af, sampled_nonaf])
rng.shuffle(sampled_indices)

df_matched = df_chap.loc[sampled_indices].reset_index(drop=True)

log.info(f'Matched cohort: {len(df_matched)} records')
log.info(f'Matched AF prevalence: {df_matched["af_detected"].mean():.4f}')
log.info(f'Matched AF count: {(df_matched["af_detected"] == 1).sum()}')
log.info(f'Matched non-AF count: {(df_matched["af_detected"] == 0).sum()}')

14:43:37 [INFO] Matched cohort: 21799 records
14:43:37 [INFO] Matched AF prevalence: 0.0720
14:43:37 [INFO] Matched AF count: 1570
14:43:37 [INFO] Matched non-AF count: 20229


## 5. Reassign folds 1–10

Drop the original `strat_fold` (which was assigned on the full Chapman cohort) and re-stratify the matched subset. Folds 1–8 train, 9 val, 10 test.

In [ ]:
# Drop original fold; reassign
df_matched = df_matched.drop(columns=['strat_fold'])

skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=FOLD_SEED)

y = df_matched['af_detected'].values
groups = df_matched['patient_id'].values
X_dummy = np.zeros((len(df_matched), 1))

fold_assignments = np.zeros(len(df_matched), dtype=int)
for fold_idx, (_, test_idx) in enumerate(skf.split(X_dummy, y, groups)):
    fold_assignments[test_idx] = fold_idx + 1

df_matched['strat_fold'] = fold_assignments

fold_counts = df_matched['strat_fold'].value_counts().sort_index()
log.info(f'Fold distribution:\n{fold_counts.to_string()}')

fold_prev = df_matched.groupby('strat_fold')['af_detected'].mean()
log.info(f'AF prevalence by fold:\n{fold_prev.round(4).to_string()}')

14:43:40 [INFO] Fold distribution:
strat_fold
1     2180
2     2180
3     2180
4     2180
5     2180
6     2180
7     2180
8     2180
9     2180
10    2179
14:43:40 [INFO] AF prevalence by fold:
strat_fold
1     0.0720
2     0.0720
3     0.0720
4     0.0720
5     0.0720
6     0.0720
7     0.0720
8     0.0720
9     0.0720
10    0.0721


## 6. Sanity checks against match targets

In [ ]:
# Sample size match
size_dev = abs(len(df_matched) - target_n) / target_n
log.info(f'Sample size deviation from PTB-XL: {size_dev:.4f}')
assert size_dev < SAMPLE_SIZE_TOLERANCE, (
    f'Sample size {len(df_matched)} differs from target {target_n} by {size_dev:.4%} '
    f'(tolerance {SAMPLE_SIZE_TOLERANCE:.0%})'
)

# Prevalence match
matched_prev = df_matched['af_detected'].mean()
prev_dev = abs(matched_prev - prev_ptb)
log.info(f'Prevalence deviation from PTB-XL: {prev_dev:.5f}')
assert prev_dev < PREVALENCE_TOLERANCE, (
    f'Prevalence {matched_prev:.4f} differs from PTB-XL {prev_ptb:.4f} by {prev_dev:.4f} '
    f'(tolerance {PREVALENCE_TOLERANCE})'
)

# Fold stratification
max_fold_dev = (fold_prev - matched_prev).abs().max()
log.info(f'Max fold prevalence deviation: {max_fold_dev:.4f}')
assert max_fold_dev < 0.02, f'Folds not stratified: max dev {max_fold_dev:.4f}'

# Patient-disjoint (each record = one patient in Chapman)
assert df_matched['patient_id'].nunique() == len(df_matched), \
    'Found duplicate patient IDs'
log.info('All match targets achieved')

14:43:40 [INFO] Sample size deviation from PTB-XL: 0.0000
14:43:40 [INFO] Prevalence deviation from PTB-XL: 0.00000
14:43:40 [INFO] Max fold prevalence deviation: 0.0000
14:43:40 [INFO] All match targets achieved


## 7. Save

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_matched.to_csv(OUTPUT_PATH, index=False)
log.info(f'Saved matched features to {OUTPUT_PATH} ({df_matched.shape[0]} rows × {df_matched.shape[1]} cols)')

# Compare schemas as a final check
ptb_cols = set(df_ptb.columns)
matched_cols = set(df_matched.columns)
if ptb_cols == matched_cols:
    log.info('Schema matches PTB-XL exactly')
else:
    log.warning(f'Schema differences:')
    log.warning(f'  Only in PTB-XL: {ptb_cols - matched_cols}')
    log.warning(f'  Only in Chapman-Matched: {matched_cols - ptb_cols}')

log.info('Done.')

14:43:41 [INFO] Saved matched features to data/chapman_matched_features.csv (21799 rows × 67 cols)
14:43:41 [INFO] Schema matches PTB-XL exactly
14:43:41 [INFO] Done.
